# Read-only audit: CUDA/T4 V-14 0.5x core artifacts

Run this only after the final 0.5x notebook reports all eight batches transferred. It verifies every artifact directory, plan binding, source/ledger binding, file and content hashes, record shape, and exact coordinate coverage.

It writes nothing. It prints only a non-scientific acceptance receipt: plan hash, artifact count, coordinate count, and index hash. It does not show or calculate any value, aggregate, rate, comparison, or conclusion.

In [ ]:
from google.colab import files, drive
from pathlib import Path
import hashlib
import json
import sys
import tarfile

sys.dont_write_bytecode = True
EXPECTED_CAPSULE_SHA256 = 'ef0681f50165615cf9a862778f80ce65b58b2b643a7fe4d2612a16dc59b71337'
uploaded = files.upload()  # Select g6_cuda_t4_v14_core_final_2x_capsule.tar.gz
assert len(uploaded) == 1
archive = Path(next(iter(uploaded)))
assert hashlib.sha256(archive.read_bytes()).hexdigest() == EXPECTED_CAPSULE_SHA256
with tarfile.open(archive, 'r:gz') as bundle:
    bundle.extractall('/content', filter='data')
ROOT = Path('/content/project')
assert (ROOT / 'research/g6/cuda_qualification/v14_core_batch_contract.py').is_file()
drive.mount('/content/gdrive')
print('Read-only audit environment ready.')

In [ ]:
PREP_LABEL = 'g6_cuda_t4_v14_core_0_5x_preparation_20260921'
RUN_LABEL = 'g6_cuda_t4_v14_core_0_5x_20260921'
PREP_DIR = Path('/content/gdrive/MyDrive') / PREP_LABEL
ARTIFACT_ROOT = Path('/content/gdrive/MyDrive') / RUN_LABEL / 'artifacts'
EXPECTED_PLAN_CONTENT_SHA256 = 'e5b44620206e5bbaca3a03e890a4f99b4f29a23b87d3e2d4f8f8be2c3067d71d'

sys.path.insert(0, str(ROOT))
from research.g6.cuda_qualification import v14_core_batch_contract as contract

plan_path = PREP_DIR / 'v14_0_5x_core_batch_plan.json'
assert plan_path.is_file() and ARTIFACT_ROOT.is_dir(), 'Preparation plan or final artifact directory is missing.'
plan = json.loads(plan_path.read_text(encoding='utf-8'))
assert plan['content_sha256'] == EXPECTED_PLAN_CONTENT_SHA256 and plan['scale'] == '0.5x'
contract.validate_plan(plan)

# A metadata-only preflight gives a field-specific report without printing any scientific record.
first_batch = plan['batches'][0]
first_dir = ARTIFACT_ROOT / first_batch['batch_id']
manifest = json.loads((first_dir / 'manifest.json').read_text(encoding='utf-8'))
records = json.loads((first_dir / 'records.json').read_text(encoding='utf-8'))
expected_record_keys = {'schema', 'batch_id', 'plan_content_sha256', 'pass_id', 'scale', 'range', 'linear_order', 'record_count', 'records', 'release_review_id', 'core_validation_identity_sha256', 'content_sha256'}
metadata_report = {
    'batch_id': first_batch['batch_id'],
    'missing_record_keys': sorted(expected_record_keys - set(records)),
    'extra_record_keys': sorted(set(records) - expected_record_keys),
    'bindings': {
        'schema': records.get('schema') == contract.core.SCHEMA_RECORDS,
        'batch_id': records.get('batch_id') == first_batch['batch_id'],
        'plan_content_sha256': records.get('plan_content_sha256') == plan['content_sha256'],
        'pass_id': records.get('pass_id') == plan['pass_id'],
        'scale': records.get('scale') == plan['scale'],
        'range': records.get('range') == manifest.get('range'),
        'linear_order': records.get('linear_order') == plan['linear_order'],
        'record_count': records.get('record_count') == first_batch['count'],
        'release_review_id': records.get('release_review_id') == manifest.get('release', {}).get('review_id'),
        'core_validation_identity_sha256': records.get('core_validation_identity_sha256') == plan['core_validation_identity_sha256'],
        'content_sha256': records.get('content_sha256') == contract.content_sha256(records),
    },
}
if metadata_report['missing_record_keys'] or metadata_report['extra_record_keys'] or not all(metadata_report['bindings'].values()):
    raise RuntimeError('0.5x batch metadata preflight failed: ' + json.dumps(metadata_report, sort_keys=True))

# Purely read-only: the index remains in memory and is not written to Drive.
receipt = contract.validate_final_set(plan, ARTIFACT_ROOT)
assert receipt['verdict'] == 'accepted'
assert receipt['batch_count'] == 8 and receipt['total_coordinates'] == 29791
print('0.5x artifact audit: PASS')
print('plan content SHA-256:', receipt['plan_content_sha256'])
print('artifact batches:', receipt['batch_count'])
print('coordinates:', receipt['total_coordinates'])
print('read-only receipt content SHA-256:', receipt['content_sha256'])